In [1]:
# Import libraries

import os
import cv2
import glob
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten, Input
from tensorflow.keras.utils import plot_model


from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Check TensorFlow version
print(tf.__version__)


2.15.0


In [2]:
main_path = './casting_data/casting_data'
train_path = os.path.join(main_path, 'train')
test_path = os.path.join(main_path, 'test')

In [3]:
# Check how many data in `train_path` and `test_path`

def check_path(path):
  labels = os.listdir(path)
  for label in labels:
    num_data = len(os.listdir(os.path.join(path, label)))
    print(f'Total Data - {label} : {num_data}')

print('Train Path')
check_path(train_path)
print('')

print('Test Path')
check_path(test_path)
print('')

Train Path
Total Data - def_front : 3758
Total Data - ok_front : 2875

Test Path
Total Data - def_front : 453
Total Data - ok_front : 262



In [4]:
# DATA AUGMENTATION + PREPROCESSING MOBILENETV2


from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# Hyperparameter
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  
SEED = 42

# TRAIN Generator (augmentasi tetap sama)
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2,
    
    # MOBILENET
    preprocessing_function=preprocess_input
)

# VALIDATION Generator
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

# TEST Generator
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# TRAIN
train_gen = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=SEED
)

# VALIDATION
val_gen = val_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED
)

# TEST
test_gen = test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    seed=SEED
)

# INFO
print(f"Train batches: {len(train_gen)}")
print(f"Val batches: {len(val_gen)}")
print(f"Test batches: {len(test_gen)}")
print(f"Class indices: {train_gen.class_indices}")

Found 5307 images belonging to 2 classes.
Found 1326 images belonging to 2 classes.
Found 715 images belonging to 2 classes.
Train batches: 166
Val batches: 42
Test batches: 23
Class indices: {'def_front': 0, 'ok_front': 1}


In [5]:
# MODEL DEFINITION - MOBILENETV2 (TRANSFER LEARNING)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Load base model (pretrained ImageNet)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze semua layer (biar ringan)
for layer in base_model.layers:
    layer.trainable = False

# Model 
model_2 = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile
from tensorflow.keras.optimizers import Adam

model_2.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

# Summary
model_2.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 2422081 (9.24 MB)
Trainable params: 164097 (641.00 KB)
Non-trainable params: 2257984 (8.61 MB)
_________________________________________________________________


In [6]:
# TRAINING MODEL MOBILENETV2

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks (biar tidak overfitting & lebih stabil)
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Training 
history_2 = model_2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20


166/166 [==============================] - 327s 2s/step - loss: 0.3618 - accuracy: 0.8623 - val_loss: 0.2490 - val_accuracy: 0.9246 - lr: 1.0000e-04
Epoch 2/20
166/166 [==============================] - 173s 1s/step - loss: 0.1996 - accuracy: 0.9365 - val_loss: 0.1737 - val_accuracy: 0.9502 - lr: 1.0000e-04
Epoch 3/20
166/166 [==============================] - 173s 1s/step - loss: 0.1493 - accuracy: 0.9565 - val_loss: 0.1327 - val_accuracy: 0.9691 - lr: 1.0000e-04
Epoch 4/20
166/166 [==============================] - 189s 1s/step - loss: 0.1169 - accuracy: 0.9683 - val_loss: 0.1075 - val_accuracy: 0.9789 - lr: 1.0000e-04
Epoch 5/20
166/166 [==============================] - 183s 1s/step - loss: 0.0926 - accuracy: 0.9791 - val_loss: 0.0904 - val_accuracy: 0.9819 - lr: 1.0000e-04
Epoch 6/20
166/166 [==============================] - 172s 1s/step - loss: 0.0792 - accuracy: 0.9825 - val_loss: 0.0793 - val_accuracy: 0.9834 - lr: 1.0000e-04
Epoch 7/20
166/166 [==================

In [7]:
import tensorflow as tf
tf.saved_model.save(model_2, 'saved_model_hf')

INFO:tensorflow:Assets written to: saved_model_hf\assets


INFO:tensorflow:Assets written to: saved_model_hf\assets


In [ ]:
import tensorflow as tf

# Dapatkan input tensor pertama
input_tensor = model_2.inputs[0]
input_name = input_tensor.name.split(':')[0]  # misal 'mobilenetv2_1.00_224_input'



In [9]:
# Simpan sebagai folder baru
tf.saved_model.save(model_2, 'saved_model_hf_v2')

INFO:tensorflow:Assets written to: saved_model_hf_v2\assets


INFO:tensorflow:Assets written to: saved_model_hf_v2\assets


In [10]:
import json
class_indices = train_gen.class_indices  # {'def_front': 0, 'ok_front': 1}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f)
print(" Class indices saved")

 Class indices saved
